In [1]:
!pip install mittens gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 41.4 MB/s eta 0:00:00


In [2]:
import nltk
from nltk.corpus import brown

from mittens import GloVe
import numpy as np

from collections import Counter, defaultdict

from gensim.models import Word2Vec, FastText, KeyedVectors

In [3]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("brown")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [4]:
def tokenize_sentence(text):
    return [t.lower() for t in nltk.word_tokenize(text)]

In [7]:
group1 = [
    "the teacher is teaching students about grammar",
    "she teaches mathematics at the university",
    "the teacher prepared teaching materials yesterday",
    "many teachers attend teaching conferences annually",
    "effective teaching requires good communication skills",
]

group2 = [
    "the unteachable student refused to learn",
]

group3 = [
    "computational linguistics combines computer science and language",
    "the computation took several hours to complete",
    "we computed the results using advanced algorithms",
    "modern computers can compute complex calculations quickly",
]

group4 = [
    "the recomputation was necessary after finding errors",
]

group5 = [
    "natural language processing is fascinating",
    "the nature of language is complex",
    "naturally occurring patterns in text are important",
]

group6 = [
    "the unnaturalness of the translation was obvious",
]

manual_sentences_raw = group1 + group2 + group3 + group4 + group5 + group6
manual_sentences_tokenized = [tokenize_sentence(s) for s in manual_sentences_raw]
all_brown_sentences = [[w.lower() for w in sent] for sent in brown.sents()]
N_BROWN_SENTENCES = 20000
brown_sentences = all_brown_sentences[:N_BROWN_SENTENCES]

In [8]:
training_sentences = brown_sentences + manual_sentences_tokenized
print(f"Number of training sentences: {len(training_sentences)}")

Number of training sentences: 20015


In [9]:
def build_limited_vocab_and_cooc(
    sentences,
    manual_sentences,
    window_size=5,
    max_vocab_size=5000,
    min_count=5,
):

    word_freq = Counter()
    for sent in sentences:
        word_freq.update(sent)

    manual_vocab = set()
    for sent in manual_sentences:
        manual_vocab.update(sent)

    most_common = word_freq.most_common()

    frequent_words = []
    for w, c in most_common:
        if w in manual_vocab:
            continue
        if c < min_count:
            continue
        frequent_words.append(w)
        if len(frequent_words) >= max_vocab_size - len(manual_vocab):
            break

    vocab = sorted(manual_vocab.union(frequent_words))
    word2idx = {w: i for i, w in enumerate(vocab)}

    print(f"Limited vocabulary size (including manual words): {len(vocab)}")

    cooc_dict = defaultdict(Counter)

    for sent in sentences:
        length = len(sent)
        for i, center_word in enumerate(sent):
            if center_word not in word2idx:
                continue
            center_idx = word2idx[center_word]

            start = max(0, i - window_size)
            end = min(length, i + window_size + 1)

            for j in range(start, end):
                if j == i:
                    continue
                context_word = sent[j]
                if context_word not in word2idx:
                    continue

                context_idx = word2idx[context_word]
                distance = abs(j - i)
                if distance == 0:
                    continue

                weight = 1.0 / distance
                cooc_dict[center_idx][context_idx] += weight

    vocab_size = len(vocab)
    cooc_matrix = np.zeros((vocab_size, vocab_size), dtype=np.float32)

    for i, context_counts in cooc_dict.items():
        for j, v in context_counts.items():
            cooc_matrix[i, j] = v

    return vocab, word2idx, cooc_matrix

In [10]:
window_size = 5
MAX_VOCAB_SIZE = 5000
MIN_COUNT = 5

vocab, word2idx, cooc_matrix = build_limited_vocab_and_cooc(
    training_sentences,
    manual_sentences_tokenized,
    window_size=window_size,
    max_vocab_size=MAX_VOCAB_SIZE,
    min_count=MIN_COUNT,
)

print(f"Co-occurrence matrix shape: {cooc_matrix.shape}")

Limited vocabulary size (including manual words): 5000
Co-occurrence matrix shape: (5000, 5000)


In [11]:
glove_dim = 100
glove_epochs = 10
learning_rate = 0.05

glove_model = GloVe(
    n=glove_dim,
    max_iter=glove_epochs,
    learning_rate=learning_rate,
)

glove_embeddings = glove_model.fit(cooc_matrix)

print("GloVe training completed.")
print("Embeddings shape:", glove_embeddings.shape)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Iteration 10: loss: 10089.626953125

GloVe training completed.
Embeddings shape: (5000, 100)


In [12]:
glove_kv = KeyedVectors(vector_size=glove_dim)
glove_kv.add_vectors(vocab, glove_embeddings)

In [13]:
glove_kv.save("brown_glove_mittens.kv")
print("Saved GloVe embeddings as 'brown_glove_mittens.kv'.")

Saved GloVe embeddings as 'brown_glove_mittens.kv'.


In [14]:
from gensim.models import Word2Vec, FastText

word2vec_model = Word2Vec(
    sentences=training_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10
)

fasttext_model = FastText(
    sentences=training_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=10
)

word2vec_model.save("brown_word2vec.model")
fasttext_model.save("brown_fasttext.model")

print("Word2Vec and FastText models trained and saved successfully.")


Word2Vec and FastText models trained and saved successfully.


In [35]:
word2vec_model = Word2Vec.load("brown_word2vec.model")
fasttext_model = FastText.load("brown_fasttext.model")

In [52]:
def get_kv(model):
    if model is None:
        return None
    if hasattr(model, "wv"):
        return model.wv
    return model

def get_vector(model, word):
    kv = get_kv(model)
    if kv is None:
        return None
    try:
        return kv[word]
    except KeyError:
        return None

def can_get_vector(model, word):
    return get_vector(model, word) is not None

def safe_similarity(model, w1, w2):
    kv = get_kv(model)
    if kv is None:
        return None
    try:
        return float(kv.similarity(w1, w2))
    except KeyError:
        return None

def safe_most_similar(model, word, topn=5):
    kv = get_kv(model)
    if kv is None:
        return None
    try:
        return kv.most_similar(word, topn=topn)
    except KeyError:
        return None

def analogy(model, a, b, c, topn=5):
    kv = get_kv(model)
    if kv is None:
        return "model_not_loaded"
    try:
        return kv.most_similar(
            positive=[b, c],
            negative=[a],
            topn=topn,
        )
    except KeyError:
        return None

def print_header(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)



In [46]:
print_header("Experiment 1: OOV (Out-of-Vocabulary) Behavior")

oov_words = ["teachable", "unteacher", "supercomputer", "miscomputation", "unnaturally"]

for w in oov_words:
    glove_has = can_get_vector(glove_kv, w)
    w2v_has = can_get_vector(word2vec_model, w)
    ft_has = can_get_vector(fasttext_model, w)

    print(f"Word: '{w}'")
    print(f"  GloVe     vector available? {glove_has}")
    print(f"  Word2Vec  vector available? {w2v_has}")
    print(f"  FastText  vector available? {ft_has}")
    print("-" * 40)


Experiment 1: OOV (Out-of-Vocabulary) Behavior
Word: 'teachable'
  GloVe     vector available? False
  Word2Vec  vector available? False
  FastText  vector available? True
----------------------------------------
Word: 'unteacher'
  GloVe     vector available? False
  Word2Vec  vector available? False
  FastText  vector available? True
----------------------------------------
Word: 'supercomputer'
  GloVe     vector available? False
  Word2Vec  vector available? False
  FastText  vector available? True
----------------------------------------
Word: 'miscomputation'
  GloVe     vector available? False
  Word2Vec  vector available? False
  FastText  vector available? True
----------------------------------------
Word: 'unnaturally'
  GloVe     vector available? False
  Word2Vec  vector available? False
  FastText  vector available? True
----------------------------------------


In [47]:
print_header("Experiment 2: Rare Words Similarity")

rare_pairs = [
    ("unteachable", "teacher"),
    ("recomputation", "computation"),
    ("unnaturalness", "natural"),
]

for w_rare, w_common in rare_pairs:
    sim_glove = safe_similarity(glove_kv, w_rare, w_common)
    sim_w2v = safe_similarity(word2vec_model, w_rare, w_common) if word2vec_model is not None else "model_not_loaded"
    sim_ft = safe_similarity(fasttext_model, w_rare, w_common) if fasttext_model is not None else "model_not_loaded"

    print(f"Pair: ('{w_rare}', '{w_common}')")
    print(f"  GloVe     similarity: {sim_glove}")
    print(f"  Word2Vec  similarity: {sim_w2v}")
    print(f"  FastText  similarity: {sim_ft}")
    print("-" * 40)


Experiment 2: Rare Words Similarity
Pair: ('unteachable', 'teacher')
  GloVe     similarity: -0.12234049290418625
  Word2Vec  similarity: 0.7832229137420654
  FastText  similarity: 0.8165361881256104
----------------------------------------
Pair: ('recomputation', 'computation')
  GloVe     similarity: 0.07577286660671234
  Word2Vec  similarity: 0.43845024704933167
  FastText  similarity: 0.9837793707847595
----------------------------------------
Pair: ('unnaturalness', 'natural')
  GloVe     similarity: -0.04157435521483421
  Word2Vec  similarity: 0.7426304221153259
  FastText  similarity: 0.9204647541046143
----------------------------------------


In [48]:
print_header("Experiment 3: Morphological Relationships")

targets = ["teaching", "computation"]

for target in targets:
    print(f"Target word: '{target}'")

    glove_sim = safe_most_similar(glove_kv, target, topn=5)
    print("  GloVe most similar:")
    print(f"    {glove_sim}")

    if word2vec_model is not None:
        w2v_sim = safe_most_similar(word2vec_model, target, topn=5)
        print("  Word2Vec most similar:")
        print(f"    {w2v_sim}")
    else:
        print("  Word2Vec most similar: model_not_loaded")

    if fasttext_model is not None:
        ft_sim = safe_most_similar(fasttext_model, target, topn=5)
        print("  FastText most similar:")
        print(f"    {ft_sim}")
    else:
        print("  FastText most similar: model_not_loaded")

    print("-" * 40)



Experiment 3: Morphological Relationships
Target word: 'teaching'
  GloVe most similar:
    [('son', 0.9366248846054077), ('english', 0.9300404787063599), ('south', 0.9266204237937927), ('blue', 0.9242684245109558), ('social', 0.9224737882614136)]
  Word2Vec most similar:
    [('atoms', 0.9712984561920166), ('arrangements', 0.9655630588531494), ('institutions', 0.9651374220848083), ('communication', 0.9629479050636292), ('equal', 0.9595540165901184)]
  FastText most similar:
    [('reaching', 0.9700968265533447), ('sky-reaching', 0.9645465612411499), ('aching', 0.9605856537818909), ('far-reaching', 0.9604399800300598), ('preaching', 0.9574865102767944)]
----------------------------------------
Target word: 'computation'
  GloVe most similar:
    [('occurring', 0.49302470684051514), ("o'banion's", 0.49057653546333313), ('petitions', 0.46286702156066895), ('computers', 0.45452630519866943), ('sudden', 0.4537630081176758)]
  Word2Vec most similar:
    [('minerals', 0.9561512470245361), (

In [49]:
print_header("Experiment 2: Rare Words Similarity")

rare_pairs = [
    ("unteachable", "teacher"),
    ("recomputation", "computation"),
    ("unnaturalness", "natural"),
]

for w_rare, w_common in rare_pairs:
    sim_glove = safe_similarity(glove_kv, w_rare, w_common)
    sim_w2v = safe_similarity(word2vec_model, w_rare, w_common) if word2vec_model is not None else "model_not_loaded"
    sim_ft = safe_similarity(fasttext_model, w_rare, w_common) if fasttext_model is not None else "model_not_loaded"

    print(f"Pair: ('{w_rare}', '{w_common}')")
    print(f"  GloVe     similarity: {sim_glove}")
    print(f"  Word2Vec  similarity: {sim_w2v}")
    print(f"  FastText  similarity: {sim_ft}")
    print("-" * 40)


Experiment 2: Rare Words Similarity
Pair: ('unteachable', 'teacher')
  GloVe     similarity: -0.12234049290418625
  Word2Vec  similarity: 0.7832229137420654
  FastText  similarity: 0.8165361881256104
----------------------------------------
Pair: ('recomputation', 'computation')
  GloVe     similarity: 0.07577286660671234
  Word2Vec  similarity: 0.43845024704933167
  FastText  similarity: 0.9837793707847595
----------------------------------------
Pair: ('unnaturalness', 'natural')
  GloVe     similarity: -0.04157435521483421
  Word2Vec  similarity: 0.7426304221153259
  FastText  similarity: 0.9204647541046143
----------------------------------------


In [50]:
print_header("Experiment 3: Morphological Relationships")

targets = ["teaching", "computation"]

for target in targets:
    print(f"Target word: '{target}'")

    glove_sim = safe_most_similar(glove_kv, target, topn=5)
    print("  GloVe most similar:")
    print(f"    {glove_sim}")

    if word2vec_model is not None:
        w2v_sim = safe_most_similar(word2vec_model, target, topn=5)
        print("  Word2Vec most similar:")
        print(f"    {w2v_sim}")
    else:
        print("  Word2Vec most similar: model_not_loaded")

    if fasttext_model is not None:
        ft_sim = safe_most_similar(fasttext_model, target, topn=5)
        print("  FastText most similar:")
        print(f"    {ft_sim}")
    else:
        print("  FastText most similar: model_not_loaded")

    print("-" * 40)


Experiment 3: Morphological Relationships
Target word: 'teaching'
  GloVe most similar:
    [('son', 0.9366248846054077), ('english', 0.9300404787063599), ('south', 0.9266204237937927), ('blue', 0.9242684245109558), ('social', 0.9224737882614136)]
  Word2Vec most similar:
    [('atoms', 0.9712984561920166), ('arrangements', 0.9655630588531494), ('institutions', 0.9651374220848083), ('communication', 0.9629479050636292), ('equal', 0.9595540165901184)]
  FastText most similar:
    [('reaching', 0.9700968265533447), ('sky-reaching', 0.9645465612411499), ('aching', 0.9605856537818909), ('far-reaching', 0.9604399800300598), ('preaching', 0.9574865102767944)]
----------------------------------------
Target word: 'computation'
  GloVe most similar:
    [('occurring', 0.49302470684051514), ("o'banion's", 0.49057653546333313), ('petitions', 0.46286702156066895), ('computers', 0.45452630519866943), ('sudden', 0.4537630081176758)]
  Word2Vec most similar:
    [('minerals', 0.9561512470245361), (

In [51]:
print_header("Experiment 4: Morphological Analogies")

a1, b1, c1 = "teacher", "teaching", "computer"
print(f"Analogy: {a1} : {b1} = {c1} : ?")

glove_result_1 = analogy(glove_kv, a1, b1, c1, topn=5)
w2v_result_1 = analogy(word2vec_model, a1, b1, c1, topn=5) if word2vec_model is not None else "model_not_loaded"
ft_result_1 = analogy(fasttext_model, a1, b1, c1, topn=5) if fasttext_model is not None else "model_not_loaded"

print(f"  GloVe result:    {glove_result_1}")
print(f"  Word2Vec result: {w2v_result_1}")
print(f"  FastText result: {ft_result_1}")
print("-" * 40)

a2, b2, c2 = "natural", "naturally", "quick"
print(f"Analogy: {a2} : {b2} = {c2} : ?")

glove_result_2 = analogy(glove_kv, a2, b2, c2, topn=5)
w2v_result_2 = analogy(word2vec_model, a2, b2, c2, topn=5) if word2vec_model is not None else "model_not_loaded"
ft_result_2 = analogy(fasttext_model, a2, b2, c2, topn=5) if fasttext_model is not None else "model_not_loaded"

print(f"  GloVe result:    {glove_result_2}")
print(f"  Word2Vec result: {w2v_result_2}")
print(f"  FastText result: {ft_result_2}")
print("-" * 40)

print("\nAll experiments completed.")


Experiment 4: Morphological Analogies
Analogy: teacher : teaching = computer : ?
  GloVe result:    [('magical', 0.45588111877441406), ('hansen', 0.43031591176986694), ('claude', 0.4068352282047272), ('compute', 0.3819945454597473), ('lloyd', 0.37671956419944763)]
  Word2Vec result: [('municipal', 0.9403806328773499), ('86', 0.9348518252372742), ('seagulls', 0.9331619143486023), ('tracts', 0.9326503276824951), ('latitude', 0.9317877888679504)]
  FastText result: [('computing', 0.936852753162384), ('combating', 0.925468385219574), ('combining', 0.9253169298171997), ('consuming', 0.9151003956794739), ('constructing', 0.9150834083557129)]
----------------------------------------
Analogy: natural : naturally = quick : ?
  GloVe result:    [('23', 0.7822062969207764), ('ave.', 0.7414202094078064), ('fla.', 0.7378655672073364), ('thomas', 0.7258419394493103), ('nevertheless', 0.7183723449707031)]
  Word2Vec result: [('lyrical', 0.9474329352378845), ('delicious', 0.9420322775840759), ('inevi